# 12. Multi-Agent Orchestration (코드 기반 멀티에이전트 오케스트레이션)

**04_Handoffs**에서는 LLM이 자율적으로 핸드오프를 결정하는 **LLM 기반 오케스트레이션**을 배웠습니다.
이번에는 **코드로 직접** 에이전트 흐름을 제어하는 패턴을 학습합니다.

### LLM 기반 vs 코드 기반 오케스트레이션

| 구분 | LLM 기반 (Handoffs) | 코드 기반 |
|------|-------------------|----------|
| 제어 주체 | LLM이 자율 판단 | 개발자가 코드로 제어 |
| 예측 가능성 | 낮음 (LLM 판단에 의존) | 높음 (결정적 흐름) |
| 유연성 | 높음 | 보통 |
| 비용/속도 | 변동적 | 예측 가능 |
| 적합한 경우 | 복잡한 대화, 분류가 어려운 경우 | 파이프라인, 배치 처리, 정해진 워크플로우 |

### 주요 패턴
1. **순차 체이닝 (Sequential Chaining)**: A → B → C
2. **병렬 실행 (Parallelization)**: A, B, C 동시 실행
3. **분류 → 라우팅 (Classification + Routing)**: 분류 후 전문 에이전트에 전달

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import openai
import asyncio

Model = "gpt-5.4-mini"

## 1. 순차 체이닝 (Sequential Chaining)

에이전트A의 출력을 에이전트B의 입력으로 전달하는 파이프라인입니다.
각 단계가 순서대로 실행됩니다.

```
[작성 에이전트] → 초안 → [편집 에이전트] → 수정본 → [번역 에이전트] → 최종 결과
```

In [4]:
from agents import Agent, Runner, trace

# Step 1: 글 작성 에이전트
writer_agent = Agent(
    name="작성기",
    instructions="""당신은 기술 블로그 작성자입니다.
주어진 주제에 대해 3~4문장의 짧은 기술 소개글을 작성하세요.
전문 용어를 적절히 사용하되 이해하기 쉽게 작성하세요.""",
    model=Model,
)

# Step 2: 편집 에이전트
editor_agent = Agent(
    name="편집기",
    instructions="""당신은 전문 편집자입니다.
주어진 글을 다음 기준으로 수정하세요:
- 문법 오류 수정
- 문장 가독성 향상
- 불필요한 표현 제거
수정된 글만 출력하세요.""",
    model=Model,
)

# Step 3: 번역 에이전트
translator_agent = Agent(
    name="번역기",
    instructions="""당신은 한영 번역가입니다.
주어진 한국어 글을 자연스러운 영어로 번역하세요.
번역된 영어만 출력하세요.""",
    model=Model,
)

# 순차 체이닝 실행
with trace("순차 체이닝: 작성 → 편집 → 번역"):
    # Step 1: 글 작성
    write_result = await Runner.run(writer_agent, "에이전트 AI에 대해 소개글을 작성해주세요.")
    print(f"[Step 1] 초안:\n{write_result.final_output}\n")

    # Step 2: 편집 (이전 단계의 출력을 입력으로 전달)
    edit_result = await Runner.run(editor_agent, write_result.final_output)
    print(f"[Step 2] 편집본:\n{edit_result.final_output}\n")

    # Step 3: 번역 (이전 단계의 출력을 입력으로 전달)
    translate_result = await Runner.run(translator_agent, edit_result.final_output)
    print(f"[Step 3] 번역본:\n{translate_result.final_output}")

[Step 1] 초안:
에이전트 AI는 단순히 질문에 답하는 챗봇을 넘어, 목표를 이해하고 스스로 계획을 세워 작업을 수행하는 지능형 시스템입니다. 여러 단계의 추론과 도구 사용, 외부 API 연동을 통해 검색, 실행, 검증까지 자동화할 수 있어 업무 효율을 크게 높입니다. 특히 반복적인 업무나 복잡한 의사결정 보조에 강점이 있으며, 최근에는 업무 자동화와 개인 비서형 서비스의 핵심 기술로 주목받고 있습니다.

[Step 2] 편집본:
에이전트 AI는 단순히 질문에 답하는 챗봇을 넘어, 목표를 이해하고 스스로 계획을 세워 작업을 수행하는 지능형 시스템입니다. 여러 단계의 추론과 도구 사용, 외부 API 연동을 통해 검색, 실행, 검증까지 자동화할 수 있어 업무 효율을 크게 높입니다. 특히 반복적인 업무와 복잡한 의사결정 보조에 강점이 있으며, 최근에는 업무 자동화와 개인 비서형 서비스의 핵심 기술로 주목받고 있습니다.

[Step 3] 번역본:
Agentic AI is an intelligent system that goes beyond a simple chatbot answering questions; it understands goals and autonomously creates plans and carries out tasks. Through multi-step reasoning, tool use, and integration with external APIs, it can automate everything from searching and execution to verification, significantly improving work efficiency. It is especially strong in repetitive tasks and supporting complex decision-making, and has recently been attracting attention as a core technology for workflow auto

## 2. 병렬 실행 (Parallelization)

서로 독립적인 에이전트를 `asyncio.gather()`로 **동시에** 실행하여 시간을 절약합니다.

```
        ┌→ [감성 분석 에이전트] → 결과1   ─┐
[입력] ──┼→ [키워드 추출 에이전트] → 결과2 ─┼→ [종합]
        └→ [요약 에이전트] → 결과3   ─────┘
```

In [5]:
# 3개의 독립적인 분석 에이전트
sentiment_agent = Agent(
    name="감성 분석기",
    instructions="""주어진 텍스트의 감성을 분석하세요.
결과 형식: "감성: [긍정/부정/중립], 신뢰도: [높음/보통/낮음], 이유: [간단한 설명]" """,
    model=Model,
)

keyword_agent = Agent(
    name="키워드 추출기",
    instructions="""주어진 텍스트에서 핵심 키워드 3~5개를 추출하세요.
결과 형식: "키워드: [키워드1], [키워드2], [키워드3], ..." """,
    model=Model,
)

summary_agent = Agent(
    name="요약기",
    instructions="""주어진 텍스트를 한 문장으로 요약하세요.
결과 형식: "요약: [한 문장 요약]" """,
    model=Model,
)

text_to_analyze = """
OpenAI의 Agent SDK는 개발자들이 AI 에이전트를 쉽게 만들 수 있도록 설계되었습니다.
핸드오프, 도구 사용, 가드레일 등의 기능을 제공하여 복잡한 AI 워크플로우를
간단한 코드로 구현할 수 있게 해줍니다. 특히 Python 개발자에게 친숙한 인터페이스를
제공하며, 프로덕션 환경에서의 안정성도 고려되어 있습니다.
"""

# asyncio.gather로 3개 에이전트를 동시에 실행
with trace("병렬 분석"):
    sentiment_result, keyword_result, summary_result = await asyncio.gather(
        Runner.run(sentiment_agent, text_to_analyze),
        Runner.run(keyword_agent, text_to_analyze),
        Runner.run(summary_agent, text_to_analyze),
    )

print(f"{sentiment_result.final_output}")
print(f"{keyword_result.final_output}")
print(f"{summary_result.final_output}")

감성: 긍정, 신뢰도: 높음, 이유: OpenAI의 Agent SDK의 장점과 편의성, 안정성을 설명하는 내용으로 전반적으로 호의적인 표현이 사용되었습니다.
키워드: OpenAI Agent SDK, AI 에이전트, 도구 사용, 가드레일, Python 인터페이스
요약: OpenAI의 Agent SDK는 핸드오프, 도구 사용, 가드레일 등을 통해 Python 개발자가 복잡한 AI 에이전트를 쉽게 만들고 프로덕션 환경에서 안정적으로 운영할 수 있게 해줍니다.


## 3. 분류 → 라우팅 (Classification + Routing)

**Structured Output**으로 입력을 분류한 후, 결과에 따라 적절한 전문 에이전트를 실행합니다.

```
[입력] → [분류 에이전트] → 카테고리 → [전문 에이전트 선택] → [전문 에이전트 실행] → [응답]
```

In [6]:
from pydantic import BaseModel

# 분류 결과를 위한 Pydantic 모델
class RequestClassification(BaseModel):
    category: str  # "기술지원", "결제문의", "일반문의"
    confidence: float  # 0.0 ~ 1.0
    reason: str

# 분류 에이전트
classifier_agent = Agent(
    name="요청 분류기",
    instructions="""고객의 요청을 다음 카테고리 중 하나로 분류하세요:
- "기술지원": 소프트웨어 오류, 설치 문제, 기술적 질문
- "결제문의": 요금, 환불, 결제 수단, 구독 관련
- "일반문의": 영업시간, 위치, 서비스 소개 등 일반 질문

category에 정확한 카테고리명을 넣고, confidence에 확신도(0~1)를 넣으세요.""",
    model=Model,
    output_type=RequestClassification,
)

# 전문 에이전트들
tech_support_agent = Agent(
    name="기술지원 전문가",
    instructions="당신은 기술지원 전문가입니다. 소프트웨어 문제를 해결하고 기술적 도움을 제공합니다.",
    model=Model,
)

billing_agent = Agent(
    name="결제 전문가",
    instructions="당신은 결제 전문가입니다. 요금, 환불, 구독 관련 질문에 답변합니다.",
    model=Model,
)

general_agent = Agent(
    name="일반 상담사",
    instructions="당신은 일반 상담사입니다. 서비스에 대한 일반적인 질문에 답변합니다.",
    model=Model,
)

# 카테고리 → 에이전트 매핑
agent_map = {
    "기술지원": tech_support_agent,
    "결제문의": billing_agent,
    "일반문의": general_agent,
}

In [7]:
# 분류 → 라우팅 실행
async def handle_customer_request(request: str):
    with trace("분류 → 라우팅"):
        # Step 1: 분류
        classification_result = await Runner.run(classifier_agent, request)
        classification = classification_result.final_output
        print(f"분류 결과: {classification.category} (확신도: {classification.confidence:.0%})")
        print(f"   이유: {classification.reason}")

        # Step 2: 라우팅 - 분류 결과에 따라 적절한 에이전트 선택
        selected_agent = agent_map.get(classification.category, general_agent)
        print(f"선택된 에이전트: {selected_agent.name}\n")

        # Step 3: 전문 에이전트 실행
        response = await Runner.run(selected_agent, request)
        print(f"응답: {response.final_output}")

    return response.final_output

In [9]:
# 테스트 1: 기술지원 질문
await handle_customer_request("프로그램이 갑자기 멈추고 에러 메시지가 떠요. 어떻게 해야 하나요?")

분류 결과: 기술지원 (확신도: 99%)
   이유: 프로그램 멈춤과 에러 메시지 발생은 소프트웨어 오류/기술 문제에 해당합니다.
선택된 에이전트: 기술지원 전문가

응답: 갑자기 멈추고 에러가 뜨면, 아래 순서로 해보세요.

1. **에러 메시지 확인**
   - 메시지 내용을 그대로 적어두세요.
   - 가능하면 **스크린샷**도 저장하세요.

2. **프로그램 재실행**
   - 프로그램을 완전히 종료한 뒤 다시 실행해 보세요.
   - 안 되면 **컴퓨터도 재부팅**해 보세요.

3. **최근 변경사항 확인**
   - 최근에 **설치한 프로그램, 업데이트, 플러그인**이 있으면 원인일 수 있어요.
   - 가능하면 최근 변경을 되돌려 보세요.

4. **권한/용량 확인**
   - 저장 공간이 부족하지 않은지 확인하세요.
   - 관리자 권한이 필요한 프로그램이면 **관리자 권한으로 실행**해 보세요.

5. **업데이트 확인**
   - 프로그램과 운영체제를 최신 버전으로 업데이트하세요.

6. **복구/재설치**
   - 특정 프로그램만 문제면 **복구 기능**을 사용하거나 **재설치**해 보세요.

7. **로그 확인**
   - 프로그램의 **로그 파일**이나 오류 코드가 있으면 원인 파악에 도움이 됩니다.

원하시면 제가 더 정확히 도와드릴게요.  
**에러 메시지 문구**, **프로그램 이름**, **사용 중인 OS(윈도우/맥)** 를 알려주세요.


'갑자기 멈추고 에러가 뜨면, 아래 순서로 해보세요.\n\n1. **에러 메시지 확인**\n   - 메시지 내용을 그대로 적어두세요.\n   - 가능하면 **스크린샷**도 저장하세요.\n\n2. **프로그램 재실행**\n   - 프로그램을 완전히 종료한 뒤 다시 실행해 보세요.\n   - 안 되면 **컴퓨터도 재부팅**해 보세요.\n\n3. **최근 변경사항 확인**\n   - 최근에 **설치한 프로그램, 업데이트, 플러그인**이 있으면 원인일 수 있어요.\n   - 가능하면 최근 변경을 되돌려 보세요.\n\n4. **권한/용량 확인**\n   - 저장 공간이 부족하지 않은지 확인하세요.\n   - 관리자 권한이 필요한 프로그램이면 **관리자 권한으로 실행**해 보세요.\n\n5. **업데이트 확인**\n   - 프로그램과 운영체제를 최신 버전으로 업데이트하세요.\n\n6. **복구/재설치**\n   - 특정 프로그램만 문제면 **복구 기능**을 사용하거나 **재설치**해 보세요.\n\n7. **로그 확인**\n   - 프로그램의 **로그 파일**이나 오류 코드가 있으면 원인 파악에 도움이 됩니다.\n\n원하시면 제가 더 정확히 도와드릴게요.  \n**에러 메시지 문구**, **프로그램 이름**, **사용 중인 OS(윈도우/맥)** 를 알려주세요.'

In [10]:
# 테스트 2: 결제 질문
await handle_customer_request("지난달 결제가 두 번 된 것 같아요. 환불 가능한가요?")

분류 결과: 결제문의 (확신도: 99%)
   이유: 중복 결제와 환불 가능 여부를 문의하고 있어 결제 관련 요청입니다.
선택된 에이전트: 결제 전문가

응답: 가능합니다. 다만 **결제 내역 확인 후 중복 결제**로 확인돼야 환불 처리할 수 있어요.

확인해드릴 정보:
- 결제일자 2건
- 결제 금액
- 결제 수단(카드/계좌/간편결제)
- 주문번호 또는 영수증 캡처

보내주시면 중복 결제 여부를 확인해드리고, 가능하면 **환불 진행 방법**까지 안내드릴게요.


'가능합니다. 다만 **결제 내역 확인 후 중복 결제**로 확인돼야 환불 처리할 수 있어요.\n\n확인해드릴 정보:\n- 결제일자 2건\n- 결제 금액\n- 결제 수단(카드/계좌/간편결제)\n- 주문번호 또는 영수증 캡처\n\n보내주시면 중복 결제 여부를 확인해드리고, 가능하면 **환불 진행 방법**까지 안내드릴게요.'

In [11]:
# 테스트 3: 일반 질문
await handle_customer_request("주말에도 고객센터가 운영되나요?")

분류 결과: 일반문의 (확신도: 98%)
   이유: 고객센터 운영 시간에 대한 문의로, 영업시간/운영 여부를 묻는 일반 질문에 해당합니다.
선택된 에이전트: 일반 상담사

응답: 주말 운영 여부는 서비스마다 달라요.  
보통은 **평일만 운영**하거나, **주말에는 제한 운영**하는 경우가 많습니다.

원하시면 제가 **운영시간 확인 방법**이나 **문의 가능한 채널**을 바로 안내해드릴게요.


'주말 운영 여부는 서비스마다 달라요.  \n보통은 **평일만 운영**하거나, **주말에는 제한 운영**하는 경우가 많습니다.\n\n원하시면 제가 **운영시간 확인 방법**이나 **문의 가능한 채널**을 바로 안내해드릴게요.'

## 4. 종합 패턴: 병렬 + 순차 조합

실전에서는 여러 패턴을 **조합**하여 사용합니다.

```
[고객 리뷰] ──→ [병렬 분석] ──→ [종합 에이전트] ──→ [최종 보고서]
                 ├ 감성 분석
                 ├ 키워드 추출
                 └ 요약
```

In [12]:
# 종합 보고서를 작성하는 에이전트
report_agent = Agent(
    name="보고서 작성기",
    instructions="""당신은 분석 보고서 작성자입니다.
주어진 분석 결과들을 종합하여 간결한 보고서를 작성하세요.
보고서 형식:
---
[분석 보고서]
- 요약: ...
- 감성: ...
- 핵심 키워드: ...
- 종합 의견: (위 분석을 바탕으로 한 줄 의견)
---""",
    model=Model,
)

customer_review = """
이 제품 정말 좋아요! 배송도 빠르고 포장도 꼼꼼했습니다.
다만 설명서가 영어로만 되어 있어서 처음에 좀 헷갈렸어요.
전체적으로는 가격 대비 성능이 훌륭하고, 다음에도 이 브랜드를 구매할 의향이 있습니다.
"""

with trace("종합 분석 파이프라인"):
    # Step 1: 병렬 분석 (동시 실행)
    print("Step 1: 병렬 분석 실행 중...\n")
    s_result, k_result, sm_result = await asyncio.gather(
        Runner.run(sentiment_agent, customer_review),
        Runner.run(keyword_agent, customer_review),
        Runner.run(summary_agent, customer_review),
    )

    print(f"  감성: {s_result.final_output}")
    print(f"  키워드: {k_result.final_output}")
    print(f"  요약: {sm_result.final_output}")

    # Step 2: 순차 체이닝 - 분석 결과를 종합
    print(f"\nStep 2: 종합 보고서 작성 중...\n")
    combined_input = f"""다음 분석 결과를 종합해주세요:

원본 리뷰: {customer_review}

분석 결과:
1. {s_result.final_output}
2. {k_result.final_output}
3. {sm_result.final_output}
"""

report_result = await Runner.run(report_agent, combined_input)
print(f"📊 최종 보고서:\n{report_result.final_output}")

Step 1: 병렬 분석 실행 중...

  감성: 감성: 긍정, 신뢰도: 높음, 이유: 전반적으로 제품 만족도가 높고 추천 의향도 있어 긍정적이며, 일부 불편함이 있지만 전체 평가를 크게 해치지 않습니다.
  키워드: 키워드: 배송, 포장, 설명서, 가격 대비 성능, 브랜드
  요약: 요약: 배송과 포장은 만족스러웠고 설명서는 아쉬웠지만, 가격 대비 성능이 좋아 다음에도 구매할 의향이 있는 제품입니다.

Step 2: 종합 보고서 작성 중...

📊 최종 보고서:
---
[분석 보고서]
- 요약: 배송과 포장은 만족스러웠고 설명서는 아쉬웠지만, 가격 대비 성능이 좋아 재구매 의향이 있는 제품입니다.
- 감성: 긍정
- 핵심 키워드: 배송, 포장, 설명서, 가격 대비 성능, 브랜드
- 종합 의견: 전반적으로 만족도가 높고 작은 불편함보다 제품 가치가 더 크게 느껴지는 긍정적 리뷰입니다.
---


### 실습 문제

**뉴스 기사 브리핑 파이프라인 (병렬 분석 + 순차 종합 패턴)**

본문 4번 패턴처럼 **병렬 분석 → 순차 종합** 구조로 뉴스 기사 브리핑 시스템을 만드세요.

1. **병렬 분석 (asyncio.gather)**: 세 에이전트를 동시에 실행하세요.
   - **제목 생성기**: 기사에 어울리는 제목 1개 생성
   - **핵심 문장 추출기**: 기사에서 가장 중요한 문장 2개 추출
   - **카테고리 분류기**: `경제 / 기술 / 사회` 중 하나로 분류

2. **순차 종합**: **브리핑 작성기** 에이전트가 세 분석 결과를 입력으로 받아
   `제목 / 카테고리 / 핵심 내용 / 한 줄 평` 형식의 **3~4줄 브리핑**을 작성하세요.

3. 전체 파이프라인을 `trace("뉴스 브리핑 파이프라인")`으로 감싸세요.

### 테스트 입력 예시

```
국내 연구진이 차세대 이차전지인 전고체 배터리의 수명을 3배 늘리는
신소재를 개발했다. 이번 기술은 전기차 주행거리 향상에 크게 기여할 것으로
기대되며, 연구팀은 3년 내 상용화를 목표로 국내 배터리 기업들과
협력을 논의 중이라고 밝혔다.
```

👉 세 분석 결과가 동시에 출력된 뒤, 이를 종합한 최종 브리핑이 출력되어야 합니다.
